# Define Catalog and Schema

In [0]:
%sql
USE CATALOG github_lakehouse;
USE SCHEMA silver;

## Create the `fork_events` Silver Table

This step creates the Delta table used to store cleaned and structured **ForkEvent** records in the Silver layer.

For the requirements of this project, no additional ForkEvent-specific payload fields are retained. The common event fields already provide the information needed to identify:

- when the fork activity occurred,
- which user performed the activity,
- and which repository was involved.

The table therefore contains three groups of fields:

- **Common event fields** — Event ID, timestamp, actor information, and repository information.
- **Bronze lineage fields** — Metadata retained from the Bronze layer so every Silver record remains traceable to its source file and ingestion.
- **Quality metadata** — `quarantine_warning`, a readable list of non-fatal problems found in the row (for example `MISSING: actor_id`). It is `NULL` for a clean row. This column is a project addition that the Silver schema reference does not list; `push_events` uses the same column.

Keeping only the required fields avoids carrying unnecessary nested payload data into the Silver layer.

The table is stored using **Delta Lake** (`USING DELTA`).

`CREATE TABLE IF NOT EXISTS` allows this step to be rerun safely without replacing the existing table.

> **Note:** ForkEvent has no retained event-specific columns in the current Silver contract. This is intentional and reflects the requirements of the project rather than the complete structure available in the original GitHub payload. `payload.forkee` is not parsed, so a missing or unusual `forkee` never sends a row to quarantine.

### ForkEvent data contract

| Rule | Fields | Result |
|---|---|---|
| Hard quarantine | JSON parse error; `event_id` or `event_type` missing; `repo_id` missing or not a number | Row goes to `quarantine.quarantine_fork_event`, not to Silver |
| Contract check | `event_type` is not `ForkEvent` | Row goes to quarantine (`INVALID_EVENT_TYPE`) |
| Keep + warn | `event_timestamp` or `actor_id` missing or not convertible; `actor_login`, `repo_name` or a lineage field missing | Row stays in Silver; the reason is added to `quarantine_warning` |
| Duplicates | Same `event_id` more than once in one batch | One row is kept: latest `_ingested_at`, then `_source_file`, then a hash of `raw_json` |

The quarantine table is created in `10_quarantine_table`. Run that notebook before this one.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.fork_events (
    -- Common fields
    event_id         STRING,
    event_type       STRING,
    event_timestamp  TIMESTAMP,
    event_date       DATE,
    actor_id         BIGINT,
    actor_login      STRING,
    repo_id          BIGINT,
    repo_name        STRING,

    -- Bronze lineage
    _source_file     STRING,
    _source_date     DATE,
    _ingested_at     TIMESTAMP,

    -- Quarantine Warning
    quarantine_warning STRING
)
USING DELTA;

## Prepare `ForkEvent` Records from the Bronze Layer

This step reads the run parameters, chooses the tables, reads Bronze, keeps only `ForkEvent` rows and parses their JSON.

### 1. Run parameters (widgets)

| Widget | Blank (default) | With a value |
|---|---|---|
| `source_date` | Process **all** source dates (backfill mode) | `YYYY-MM-DD`: process only that `_source_date`. Any other format, or an impossible date, fails the run. |
| `test_schema` | Production tables | `test_<name>`: every table is read from and written to `github_lakehouse.test_<name>` |
| `run_id` (audit cell) | A generated `manual-<uuid>` | The Job passes `{{job.run_id}}` |

The four table names (`BRONZE_TABLE`, `SILVER_TABLE`, `QUARANTINE_TABLE`, `AUDIT_TABLE`) are set **only** in this cell. Later cells must never redefine them, otherwise a test run could write to production.

### 2. Filter first, then parse

`get_json_object(raw_json, '$.type') = 'ForkEvent'` keeps only ForkEvent rows **before** the full JSON is parsed, so Spark does not parse the other event types.

The parse schema contains only the common fields. `payload` is not in the schema, because ForkEvent keeps no payload fields.

All values are read as **strings**, including IDs and the timestamp. Datatype conversion happens in the next step, so a bad value can be detected and reported instead of silently becoming `NULL` during parsing.

### 3. Parse permissively

`from_json()` runs in **PERMISSIVE** mode. A record that does not match the schema (for example `actor` is text instead of an object) does not stop the run. Its text is placed in `_corrupt_record`, and the row is quarantined later as `JSON_PARSE_ERROR`.

### 4. Evidence kept for later steps

- `raw_json` — the original event, for quarantine and debugging.
- `_corrupt_record` — set when parsing failed.

These two columns never reach the Silver table.

> **Note:** Nothing is validated or written in this step.

In [0]:
import re
from datetime import datetime, timezone

pipeline_run_started_at = datetime.now(timezone.utc)

from pyspark.sql import functions as F

# The one event type this notebook handles. Used by the Bronze filter,
# the validation rule and the audit row.
EVENT_TYPE = "ForkEvent"


# 1. Optional batch-date parameter.
#    Blank       -> process all source dates (intentional backfill mode).
#    YYYY-MM-DD  -> process only that _source_date.
#    Anything else fails the run instead of silently matching zero rows.
dbutils.widgets.text("source_date", "")

source_date = dbutils.widgets.get("source_date").strip()
source_date_value = None

if source_date:
    # Shape check first: strptime alone would also accept "2024-2-1".
    if not re.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}", source_date):
        raise ValueError(
            f"Invalid source_date '{source_date}': "
            "expected YYYY-MM-DD (for example 2024-02-01), "
            "or leave it blank to process all dates."
        )

    # Calendar check: rejects impossible dates such as 2024-02-31.
    try:
        source_date_value = datetime.strptime(source_date, "%Y-%m-%d").date()
    except ValueError:
        raise ValueError(
            f"Invalid source_date '{source_date}': "
            "not a real calendar date."
        ) from None

print(
    f"Processing scope: source_date = {source_date_value}"
    if source_date_value
    else "Processing scope: ALL source dates (source_date is blank)"
)

# Table targets.
#    Blank test_schema -> production tables (default; used by the Job).
#    test_<name>       -> every table is read/written in github_lakehouse.test_<name>.
dbutils.widgets.text("test_schema", "")
test_schema = dbutils.widgets.get("test_schema").strip()

if test_schema:
    if not re.fullmatch(r"test_[a-z0-9_]+", test_schema):
        raise ValueError(
            f"Invalid test_schema '{test_schema}': must match test_[a-z0-9_]+"
        )
    BRONZE_TABLE = f"github_lakehouse.{test_schema}.github_events_raw"
    SILVER_TABLE = f"github_lakehouse.{test_schema}.fork_events"
    QUARANTINE_TABLE = f"github_lakehouse.{test_schema}.quarantine_fork_event"
    AUDIT_TABLE = f"github_lakehouse.{test_schema}.silver_pipeline_runs"
else:
    BRONZE_TABLE = "github_lakehouse.bronze.github_events_raw"
    SILVER_TABLE = "github_lakehouse.silver.fork_events"
    QUARANTINE_TABLE = "github_lakehouse.quarantine.quarantine_fork_event"
    AUDIT_TABLE = "github_lakehouse.audit.silver_pipeline_runs"

print(f"Tables: {BRONZE_TABLE} | {SILVER_TABLE} | {QUARANTINE_TABLE} | {AUDIT_TABLE}")

# Read Bronze.
bronze_df = spark.table(
    BRONZE_TABLE
    ).select(
        "raw_json",
        "_source_file",
        "_source_date",
        "_ingested_at"
    )

# Filter to one source date when provided.
# A Python date literal avoids Spark's to_date() parsing (and its ANSI-mode differences).
if source_date_value:
    bronze_df = bronze_df.filter(
        F.col("_source_date") == F.lit(source_date_value)
    )

# 2. Filter ForkEvent rows before parsing the full event.
fork_bronze_df = bronze_df.filter(
    F.get_json_object(F.col("raw_json"), "$.type") == EVENT_TYPE
)


# 3. Define the source fields needed for ForkEvent.
# Only the common fields: ForkEvent keeps no payload fields, so payload
# (including payload.forkee) is deliberately not part of the schema.
# Keep values as strings until the validation and casting step.
fork_schema = """
STRUCT<
    id: STRING,
    type: STRING,
    created_at: STRING,
    actor: STRUCT<
        id: STRING,
        login: STRING
    >,
    repo: STRUCT<
        id: STRING,
        name: STRING
    >,
    _corrupt_record: STRING
>
"""


# 4. Parse the filtered JSON permissively.
parsed_df = fork_bronze_df.withColumn(
    "event",
    F.from_json(
        F.col("raw_json"),
        fork_schema,
        {
            "mode": "PERMISSIVE",
            "columnNameOfCorruptRecord": "_corrupt_record"
        }
    )
)


# 5. Extract the fields and prepare the ForkEvent DataFrame.
fork_events_df = parsed_df.select(
    F.col("event.id").alias("event_id"),
    F.col("event.type").alias("event_type"),
    F.col("event.created_at").alias("event_timestamp"),
    F.col("event.actor.id").alias("actor_id"),
    F.col("event.actor.login").alias("actor_login"),
    F.col("event.repo.id").alias("repo_id"),
    F.col("event.repo.name").alias("repo_name"),

    # Preserve the original Bronze lineage values.
    F.col("_source_file"),
    F.col("_source_date"),
    F.col("_ingested_at"),

    # Keep the raw evidence for later validation and quarantine.
    F.col("raw_json"),
    F.col("event._corrupt_record").alias("_corrupt_record")
)

## Validate and Convert `ForkEvent` Data Types

This step converts the parsed strings into the datatypes of the Silver table. It keeps the original text next to each converted value.

### 1. UTC

`spark.sql.session.timeZone = UTC`, so GitHub timestamps and the derived `event_date` are always interpreted in UTC.

### 2. Safe conversions

| Column | Conversion | Original text kept in |
|---|---|---|
| `event_timestamp` | `try_to_timestamp()` | `event_timestamp_raw` |
| `event_date` | derived from the converted `event_timestamp` | — |
| `actor_id` | `try_cast(... AS BIGINT)` | `actor_id_raw` |
| `repo_id` | `try_cast(... AS BIGINT)` | `repo_id_raw` |

`try_to_timestamp()` and `try_cast()` return `NULL` instead of failing the run when a value cannot be converted. Keeping the `_raw` column lets the next step tell two cases apart:

- the value was **missing** in the source, or
- the value was **present but invalid** (for example `repo_id = "abc"`).

### 3. Columns passed through unchanged

`event_id`, `event_type`, `actor_login`, `repo_name`, the three lineage columns, `raw_json` and `_corrupt_record`.

> **Note:** Rows are not classified yet. The `_raw` columns are temporary evidence and only reach the quarantine table.

In [0]:
from pyspark.sql import functions as F

# Use UTC for timestamps and the derived event date.
spark.conf.set("spark.sql.session.timeZone", "UTC")


# 1. Preserve the original timestamp, then convert it.
timestamp_df = (
    fork_events_df
    .withColumn(
        "event_timestamp_raw",
        F.col("event_timestamp")
    )
    .withColumn(
        "event_timestamp",
        F.try_to_timestamp(F.col("event_timestamp"))
    )
)


# 2. Convert numeric fields and preserve their original values.
fork_events_typed_df = timestamp_df.select(
    # String fields pass through directly.
    F.col("event_id"),
    F.col("event_type"),

    # Timestamp and its original value
    F.col("event_timestamp"),
    F.col("event_timestamp_raw"),

    # Derive the event date from the converted timestamp.
    F.expr("try_cast(event_timestamp AS DATE)").alias("event_date"),

    # Actor fields
    F.expr("try_cast(actor_id AS BIGINT)").alias("actor_id"),
    F.col("actor_id").alias("actor_id_raw"),
    F.col("actor_login"),

    # Repository fields
    F.expr("try_cast(repo_id AS BIGINT)").alias("repo_id"),
    F.col("repo_id").alias("repo_id_raw"),
    F.col("repo_name"),

    # Bronze lineage passes through directly.
    F.col("_source_file"),
    F.col("_source_date"),
    F.col("_ingested_at"),

    # Evidence for later validation and quarantine
    F.col("raw_json"),
    F.col("_corrupt_record")
)

## Handle `ForkEvent` NULLs and Quarantine Records

This step applies the Revised Silver NULL & Quarantine Policy to the common fields.

1. **Blank text becomes `NULL`** for `event_id`, `event_type`, `actor_login` and `repo_name`, so `""` and `"   "` are treated as missing.
2. **Hard-quarantine check** (`quarantine_reason`). The row cannot safely enter Silver when:
   - the JSON could not be parsed (`JSON_PARSE_ERROR`, which replaces any other reason), or
   - `event_id` or `event_type` is missing, or
   - `repo_id` is missing or is not a number.
3. **Warnings** (`quarantine_warning`) for problems that do **not** remove the row:
   - `event_timestamp` or `actor_id` missing or not convertible,
   - `actor_login`, `repo_name`, `_source_file`, `_source_date` or `_ingested_at` missing.

`conversion_null_reason()` writes `MISSING: <column>` when the source had no value and `INVALID_CAST: <column>` when a value was present but could not be converted.

Output DataFrames:

- `quarantine_fork_df` — hard-quarantine rows, with every evidence column. They are written to quarantine later.
- `fork_events_ready_df` — usable rows, with `quarantine_warning` (`NULL` when the row is clean).

In [0]:
from pyspark.sql import functions as F


spark.conf.set("spark.sql.session.timeZone", "UTC")

fork_df = fork_events_typed_df


# ---------------------------------------------------------
# 1. Normalize blank string fields to NULL
# ---------------------------------------------------------

for column_name in [
    "event_id",
    "event_type",
    "actor_login",
    "repo_name"
]:
    fork_df = fork_df.withColumn(
        column_name,
        F.when(
            F.col(column_name).isNull()
            | (F.trim(F.col(column_name)) == ""),
            F.lit(None)
        ).otherwise(
            F.col(column_name)
        )
    )


# ---------------------------------------------------------
# 2. Helper:
#    distinguish source NULL from failed conversion
# ---------------------------------------------------------

def conversion_null_reason(column_name):

    raw_column = F.col(column_name + "_raw")

    return F.when(
        F.col(column_name).isNull(),

        F.when(
            raw_column.isNull()
            | (F.trim(raw_column) == ""),

            F.lit("MISSING: " + column_name)

        ).otherwise(

            F.lit("INVALID_CAST: " + column_name)

        )
    )


# ---------------------------------------------------------
# 3. NULL conditions that require quarantine
# ---------------------------------------------------------


checked_df = fork_df.withColumn(
    "quarantine_reason",

    F.when(
        F.col("_corrupt_record").isNotNull(),
        F.lit("JSON_PARSE_ERROR")
    ).otherwise(
        F.concat_ws(
            "; ",

            F.when(
                F.col("event_id").isNull(),
                F.lit("MISSING: event_id")
            ),

            F.when(
                F.col("event_type").isNull(),
                F.lit("MISSING: event_type")
            ),

            conversion_null_reason("repo_id")
        )
    )
)



# ---------------------------------------------------------
# 4. Split hard-null records from usable records
# ---------------------------------------------------------

# Keep every intermediate column (raw values, raw_json, _corrupt_record,
# quarantine_reason). quarantined_at, quarantine_key and the final
# destination-column projection are added for BOTH quarantine branches
# together in the "Prepare Final Silver and Quarantine DataFrames" cell.
quarantine_fork_df = (
    checked_df
    .filter(
        F.col("quarantine_reason") != ""
    )
)


fork_events_ready_df = (
    checked_df
    .filter(
        F.col("quarantine_reason") == ""
    )
    .drop("quarantine_reason")
)


# ---------------------------------------------------------
# 5. Record allowed NULLs / conversion-null warnings
# ---------------------------------------------------------

fork_events_ready_df = (
    fork_events_ready_df
    .withColumn(
        "quarantine_warning",

        F.concat_ws(
            "; ",

            conversion_null_reason("event_timestamp"),

            conversion_null_reason("actor_id"),

            F.when(
                F.col("actor_login").isNull(),
                F.lit("MISSING: actor_login")
            ),

            F.when(
                F.col("repo_name").isNull(),
                F.lit("MISSING: repo_name")
            ),

            F.when(
                F.col("_source_file").isNull(),
                F.lit("MISSING: _source_file")
            ),

            F.when(
                F.col("_source_date").isNull(),
                F.lit("MISSING: _source_date")
            ),

            F.when(
                F.col("_ingested_at").isNull(),
                F.lit("MISSING: _ingested_at")
            )
        )
    )
)


# Empty string -> actual NULL
fork_events_ready_df = (
    fork_events_ready_df
    .withColumn(
        "quarantine_warning",

        F.when(
            F.col("quarantine_warning") != "",
            F.col("quarantine_warning")
        )
    )
)

## Validate `ForkEvent` Records

This step applies the validation rules to the rows that passed the NULL step.

ForkEvent has only one rule:

- **Contract check (hard)** — `event_type` must be `ForkEvent`. The Bronze filter already guarantees this, so the rule is defensive; a failing row would go to quarantine as `INVALID_EVENT_TYPE`.

ForkEvent keeps no event-specific fields, so there are **no** event-specific warning rules and **no** metric clean-up here. (In `01_push_events` this is where the commit counts are checked and sanitized.)

Output DataFrames:

- `validation_quarantine_fork_df` — rows that failed the contract check.
- `fork_events_validated_df` — rows that continue to deduplication, including rows that only have warnings.

In [0]:
from pyspark.sql import functions as F


# ---------------------------------------------------------
# 1. Hard validation rule
#    Defensive contract check - only ForkEvent rows belong here.
#    NULL event_type was already quarantined by the NULL cell, so skip it.
# ---------------------------------------------------------

invalid_event_type = F.when(
    F.col("event_type").isNotNull()
    & (F.col("event_type") != EVENT_TYPE),
    F.lit("INVALID_EVENT_TYPE")
)


# ForkEvent keeps no event-specific fields, so there are no
# event-specific warning rules and no metric sanitization in this cell.


# ---------------------------------------------------------
# 2. Apply the rule
#    concat_ws skips NULLs, so only failed rules appear in the text.
# ---------------------------------------------------------

fork_validation_checked_df = (
    fork_events_ready_df
    .withColumn(
        "validation_quarantine_reason",
        F.concat_ws(
            "; ",
            invalid_event_type
        )
    )
)


# ---------------------------------------------------------
# 3. Empty string -> actual NULL
#    concat_ws returns "" when nothing failed.
# ---------------------------------------------------------

fork_validation_checked_df = (
    fork_validation_checked_df
    .withColumn(
        "validation_quarantine_reason",
        F.when(
            F.col("validation_quarantine_reason") != "",
            F.col("validation_quarantine_reason")
        )
    )
)


# ---------------------------------------------------------
# 4. Split hard failures from rows that continue
# ---------------------------------------------------------

# Hard failures: keep every column (raw_json, *_raw, _corrupt_record,
# warnings) as evidence for later quarantine finalization.
validation_quarantine_fork_df = (
    fork_validation_checked_df
    .filter(
        F.col("validation_quarantine_reason").isNotNull()
    )
)

# Rows that pass hard validation continue to deduplication.
# Temporary evidence columns are still kept on purpose.
fork_events_validated_df = (
    fork_validation_checked_df
    .filter(
        F.col("validation_quarantine_reason").isNull()
    )
    .drop("validation_quarantine_reason")
)

## Deduplicate ForkEvent Records

GH Archive can contain the same event more than once (profiling found 2 extra ForkEvent rows in Feb 1–7 2024, each an identical copy).

Only **one row per `event_id`** is kept within this batch. The winner is chosen in this order:

1. latest `_ingested_at`,
2. then the largest `_source_file`,
3. then the largest SHA-256 hash of `raw_json`, which only makes the choice repeatable when everything else is equal.

Deduplication is by `event_id` only. Rows from earlier runs are handled by the insert-only MERGE in the write step.

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

w = Window.partitionBy("event_id").orderBy(
    F.col("_ingested_at").desc_nulls_last(),
    F.col("_source_file").desc_nulls_last(),
    F.sha2(F.col("raw_json"), 256).desc_nulls_last()
    )

fork_events_dedup_df = (
    fork_events_validated_df
    .withColumn(
        "_rn",
        F.row_number().over(w)
    )
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    )

## Prepare Final Silver and Quarantine DataFrames

- **Silver:** select exactly the columns of the Silver table, in table order. This drops `raw_json`, the `_raw` columns and `_corrupt_record`.
- **Quarantine:** combine the two quarantine branches (NULL step and validation step) in the quarantine table's column order, add `quarantine_key`, keep one row per key, and stamp `quarantined_at`.
- **Contract check:** both DataFrames must match their target tables exactly (names, order and datatypes) before anything is written.

`quarantine_key` is the SHA-256 of `_source_file`, `_source_date` and `raw_json`. The same bad source line therefore gets the same key on every run, which makes the quarantine write repeatable.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# Table names are defined in the "Read, filter and parse" cell.


# ---------------------------------------------------------
# Helper: fail fast if a DataFrame does not match its target table
# (same column names, same order, same datatypes). Reads metadata only.
# ---------------------------------------------------------
def assert_matches_table(df, table_name):
    expected = [
        (f.name, f.dataType.simpleString())
        for f in spark.table(table_name).schema
    ]
    actual = [
        (f.name, f.dataType.simpleString())
        for f in df.schema
    ]
    if actual != expected:
        raise ValueError(
            f"Schema mismatch for {table_name}\n"
            f"expected: {expected}\n"
            f"actual:   {actual}"
        )
    print(f"Schema OK: {table_name}")


# ---------------------------------------------------------
# 1. Final Silver DataFrame
#    Exactly the destination columns; drops raw_json, *_raw,
#    _corrupt_record and any temporary columns.
# ---------------------------------------------------------
silver_columns = spark.table(SILVER_TABLE).columns

silver_fork_final_df = fork_events_dedup_df.select(*silver_columns)


# ---------------------------------------------------------
# 2. Align the two quarantine branches
#    Both keep the same intermediate evidence columns.
#    quarantine_warning (branch 2 only) is not a quarantine column,
#    so it is not carried over.
# ---------------------------------------------------------
quarantine_columns = spark.table(QUARANTINE_TABLE).columns

# Columns both branches already have (everything except the two
# columns created below).
quarantine_evidence_columns = [
    c for c in quarantine_columns
    if c not in ("quarantined_at", "quarantine_key")
]

null_quarantine_df = quarantine_fork_df.select(
    *quarantine_evidence_columns
)

validation_quarantine_df = (
    validation_quarantine_fork_df
    .withColumnRenamed(
        "validation_quarantine_reason",
        "quarantine_reason"
    )
    .select(*quarantine_evidence_columns)
)

all_quarantine_df = null_quarantine_df.unionByName(
    validation_quarantine_df
)


# ---------------------------------------------------------
# 3. Stable quarantine key
#    to_json(struct(...)) labels every field and escapes its text,
#    so values can never run into each other, e.g.
#      ("ab", "c") and ("a", "bc") give different keys.
#    ignoreNullFields=false writes NULL as JSON null, so NULL and the
#    text "null" or "" also give different keys.
#    _ingested_at, quarantined_at and quarantine_reason are excluded.
# ---------------------------------------------------------
def quarantine_key_col():
    return F.sha2(
        F.to_json(
            F.struct(
                F.col("_source_file"),
                F.col("_source_date").cast("string").alias("_source_date"),
                F.col("raw_json")
            ),
            {"ignoreNullFields": "false"}
        ),
        256
    )


# ---------------------------------------------------------
# 4. Key, deduplicate within the batch, stamp, project
#    Same key = same source line (e.g. the same file ingested twice).
#    Keep the latest ingestion deterministically, like the Silver dedup.
# ---------------------------------------------------------
quarantine_window = Window.partitionBy("quarantine_key").orderBy(
    F.col("_ingested_at").desc_nulls_last(),
    F.col("quarantine_reason").asc_nulls_last()
)

quarantine_fork_final_df = (
    all_quarantine_df
    .withColumn("quarantine_key", quarantine_key_col())
    .withColumn("_rn", F.row_number().over(quarantine_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    .withColumn("quarantined_at", F.current_timestamp())
    .select(*quarantine_columns)
)


# ---------------------------------------------------------
# 5. Contract checks before any write (metadata only, no Spark job)
# ---------------------------------------------------------
assert_matches_table(silver_fork_final_df, SILVER_TABLE)
assert_matches_table(quarantine_fork_final_df, QUARANTINE_TABLE)

## Write ForkEvent Data to Delta Lake

Both writes are **insert-only Delta MERGEs**:

| Target | Match key | Effect of a rerun |
|---|---|---|
| quarantine table | `quarantine_key` | Rows already present are skipped |
| Silver table | `event_id` | Events already present are skipped |

Existing rows are never updated or deleted. The quarantine MERGE runs first and the Silver MERGE second. Each MERGE is one atomic Delta transaction, so if either one fails the notebook can simply be rerun.

If a write fails, the loop stops, records the error in `fork_write_results`, and lets the audit cell write a `FAILED` row before the run fails.

In [0]:
from datetime import datetime, timezone

# SILVER_TABLE / QUARANTINE_TABLE and the two *_final_df DataFrames
# come from the previous cells.


def latest_table_version(table_name):
    """Current Delta version of a table (metadata only)."""
    return spark.sql(
        f"DESCRIBE HISTORY {table_name} LIMIT 1"
    ).collect()[0]["version"]


def insert_only_merge(source_df, target_table, key_column, view_name):
    """
    Insert source rows whose key is not already in the target.
    Existing rows are never updated or deleted, so reruns are safe.
    The source must already be unique and non-NULL on key_column
    (guaranteed by the NULL-handling, dedup and final-preparation cells).
    """
    version_before = latest_table_version(target_table)
    started_at = datetime.now(timezone.utc)

    source_df.createOrReplaceTempView(view_name)

    merge_metrics = spark.sql(f"""
        MERGE INTO {target_table} AS t
        USING {view_name} AS s
        ON t.{key_column} = s.{key_column}
        WHEN NOT MATCHED THEN INSERT *
    """).collect()[0].asDict()

    finished_at = datetime.now(timezone.utc)

    inserted = int(merge_metrics.get("num_inserted_rows", 0))
    updated = int(merge_metrics.get("num_updated_rows", 0))
    deleted = int(merge_metrics.get("num_deleted_rows", 0))

    # Safety invariant: an insert-only MERGE must never change history.
    if updated or deleted:
        raise RuntimeError(
            f"{target_table}: insert-only MERGE changed existing rows "
            f"(updated={updated}, deleted={deleted})"
        )

    return {
        "status": "SUCCEEDED",
        "target_table": target_table,
        "rows_inserted": inserted,
        "version_before": version_before,
        # Unchanged version = nothing was committed (empty batch or full rerun).
        "version_after": latest_table_version(target_table),
        "started_at": started_at,
        "finished_at": finished_at,
        "error": None,
    }


# ---------------------------------------------------------
# Run both writes. Quarantine first, then Silver.
# Each MERGE is one atomic Delta transaction; if either fails,
# the whole notebook can simply be rerun.
# ---------------------------------------------------------
fork_write_plan = [
    ("quarantine", quarantine_fork_final_df, QUARANTINE_TABLE,
     "quarantine_key", "fork_quarantine_merge_src"),
    ("silver", silver_fork_final_df, SILVER_TABLE,
     "event_id", "fork_silver_merge_src"),
]

# Kept for the audit step.
fork_write_results = {
    name: {"status": "NOT_STARTED", "target_table": table}
    for name, _, table, _, _ in fork_write_plan
}

for name, df, table, key, view in fork_write_plan:
    fork_write_results[name] = {"status": "RUNNING", "target_table": table}
    try:
        fork_write_results[name] = insert_only_merge(df, table, key, view)
    except Exception as e:
        fork_write_results[name] = {
            "status": "FAILED",
            "target_table": table,
            "error": f"{type(e).__name__}: {str(e)[:1000]}",
        }
        print(f"{name}: FAILED -> {fork_write_results[name]['error']}")
        # Stop writing, but let the audit cell record the failure
        # and then fail the run.
        break

    r = fork_write_results[name]
    print(
        f"{name}: {r['status']} | inserted={r['rows_inserted']:,} | "
        f"version {r['version_before']} -> {r['version_after']}"
    )

## ForkEvent Pipeline Audit

This cell appends **exactly one row** per run to the shared audit table `audit.silver_pipeline_runs` (the same table every Silver notebook uses).

1. **Counts** come from one aggregation pass over `checked_df` (every ForkEvent row in scope, before any split): input rows, hard-quarantine rows, validation-quarantine rows, validated rows, Silver candidates and quarantine candidates.
2. **Quality metrics** (`quality_metrics` map) count, among validated rows, the rows with a NULL `event_timestamp`, `actor_id`, `actor_login` or `repo_name`, and rows with missing lineage.
3. **Reconciliation** compares those counts with the row counts that each MERGE actually received (Delta history `numSourceRows`). A mismatch marks the run `FAILED`.
4. **Status:** `SUCCEEDED`, `NO_INPUT` (no ForkEvent rows in scope; check `source_date`), or `FAILED` (a write or the reconciliation failed). A `FAILED` run raises an error after the audit row is written, so the Job task fails too.

In [0]:
import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F

PIPELINE_NAME = "silver_fork_events"
# EVENT_TYPE ("ForkEvent") is defined in the "Read, filter and parse" cell.

# A Job passes {{job.run_id}} here. A manual run gets a generated id.
dbutils.widgets.text("run_id", "")
run_id = dbutils.widgets.get("run_id").strip() or f"manual-{uuid.uuid4()}"


# ---------------------------------------------------------
# 1. Source-row count of the exact MERGE commit made by the write cell.
#    Returns None if that commit cannot be identified safely,
#    which means another writer committed in between.
# ---------------------------------------------------------
def merge_source_rows(write_result):
    if write_result.get("status") != "SUCCEEDED":
        return None
    if write_result["version_after"] != write_result["version_before"] + 1:
        return None
    commit = (
        spark.sql(f"DESCRIBE HISTORY {write_result['target_table']} LIMIT 20")
        .filter(F.col("version") == write_result["version_after"])
        .select("operation", "operationMetrics")
        .collect()
    )
    if not commit or commit[0]["operation"] != "MERGE":
        return None
    metrics = commit[0]["operationMetrics"]
    if int(metrics.get("numTargetRowsInserted", -1)) != write_result["rows_inserted"]:
        return None
    return int(metrics["numSourceRows"])


silver_w = fork_write_results["silver"]
quarantine_w = fork_write_results["quarantine"]
write_failed = any(
    r["status"] != "SUCCEEDED" for r in fork_write_results.values()
)

audit = {
    "audit_id": str(uuid.uuid4()),
    "run_id": run_id,
    "pipeline_name": PIPELINE_NAME,
    "event_type": EVENT_TYPE,
    "processing_scope": "SINGLE_DATE" if source_date_value else "ALL_DATES",
    "source_date": source_date_value,
    "run_started_at": pipeline_run_started_at,
    "silver_rows_inserted": silver_w.get("rows_inserted"),
    "silver_table_version": silver_w.get("version_after"),
    "quarantine_rows_inserted": quarantine_w.get("rows_inserted"),
    "quarantine_table_version": quarantine_w.get("version_after"),
}


if write_failed:
    # Skip the count pass: the batch did not complete.
    failed = [r for r in fork_write_results.values() if r["status"] == "FAILED"]
    audit["status"] = "FAILED"
    audit["error_message"] = failed[0]["error"] if failed else "Write did not complete"
    audit["reconciliation_passed"] = False

else:
    # -----------------------------------------------------
    # 2. ONE aggregation pass over checked_df: every ForkEvent row
    #    in scope, before any split. It reuses the rule column
    #    defined in the validation cell and quarantine_key_col()
    #    from the final-preparation cell, so the logic lives
    #    in one place.
    # -----------------------------------------------------
    def n(condition):
        return F.count(F.when(condition, True))

    is_hard_null = F.col("quarantine_reason") != ""
    is_validation_q = ~is_hard_null & invalid_event_type.isNotNull()
    is_validated = ~is_hard_null & invalid_event_type.isNull()
    is_quarantine = is_hard_null | is_validation_q

    # Measured on validated rows. ForkEvent has only common-field checks.
    quality_checks = {
        "null_event_timestamp": F.col("event_timestamp").isNull(),
        "null_actor_id": F.col("actor_id").isNull(),
        "null_actor_login": F.col("actor_login").isNull(),
        "null_repo_name": F.col("repo_name").isNull(),
        "missing_lineage":
            F.col("_source_file").isNull()
            | F.col("_source_date").isNull()
            | F.col("_ingested_at").isNull(),
    }

    counts = checked_df.agg(
        F.count(F.lit(1)).alias("bronze_input_rows"),
        n(is_hard_null).alias("hard_null_quarantine_rows"),
        n(is_validation_q).alias("validation_quarantine_rows"),
        n(is_validated).alias("validated_rows"),
        F.countDistinct(
            F.when(is_validated, F.col("event_id"))
        ).alias("silver_candidate_rows"),
        F.countDistinct(
            F.when(is_quarantine, quarantine_key_col())
        ).alias("quarantine_candidate_rows"),
        *[n(is_validated & cond).alias(name) for name, cond in quality_checks.items()],
    ).collect()[0].asDict()

    quality_metrics = {name: int(counts.pop(name)) for name in quality_checks}
    audit.update({k: int(v) for k, v in counts.items()})
    audit["quality_metrics"] = quality_metrics

    audit["duplicate_rows_removed"] = (
        audit["validated_rows"] - audit["silver_candidate_rows"]
    )
    audit["quarantine_duplicates_removed"] = (
        audit["hard_null_quarantine_rows"]
        + audit["validation_quarantine_rows"]
        - audit["quarantine_candidate_rows"]
    )
    audit["silver_rows_already_present"] = (
        audit["silver_candidate_rows"] - audit["silver_rows_inserted"]
    )

    # -----------------------------------------------------
    # 3. Reconciliation. The audit pass and the MERGE source are
    #    computed independently, so a mismatch reveals a real
    #    problem in dedup, keying or the MERGE input.
    # -----------------------------------------------------
    checks = {
        "silver candidates = Silver MERGE source rows":
            merge_source_rows(silver_w) == audit["silver_candidate_rows"],
        "quarantine candidates = quarantine MERGE source rows":
            merge_source_rows(quarantine_w) == audit["quarantine_candidate_rows"],
        "silver inserted <= silver candidates":
            audit["silver_rows_inserted"] <= audit["silver_candidate_rows"],
        "quarantine inserted <= quarantine candidates":
            audit["quarantine_rows_inserted"] <= audit["quarantine_candidate_rows"],
    }
    failed_checks = [name for name, ok in checks.items() if not ok]

    audit["reconciliation_passed"] = not failed_checks
    if failed_checks:
        audit["status"] = "FAILED"
        audit["error_message"] = "Reconciliation failed: " + "; ".join(failed_checks)
    elif audit["bronze_input_rows"] == 0:
        audit["status"] = "NO_INPUT"
        audit["error_message"] = None
    else:
        audit["status"] = "SUCCEEDED"
        audit["error_message"] = None


# ---------------------------------------------------------
# 4. Append exactly one audit row for this attempt.
# ---------------------------------------------------------
audit["run_finished_at"] = datetime.now(timezone.utc)

audit_schema = spark.table(AUDIT_TABLE).schema
audit_df = spark.createDataFrame(
    [tuple(audit.get(f.name) for f in audit_schema)],
    schema=audit_schema,
)
audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)

print(f"Audit written: run_id={run_id} status={audit['status']}")
for key in [
    "bronze_input_rows", "hard_null_quarantine_rows", "validation_quarantine_rows",
    "validated_rows", "duplicate_rows_removed", "silver_candidate_rows",
    "silver_rows_inserted", "silver_rows_already_present",
    "quarantine_candidate_rows", "quarantine_rows_inserted",
]:
    if audit.get(key) is not None:
        print(f"  {key}: {audit[key]:,}")
if audit.get("quality_metrics"):
    print("  quality_metrics:", audit["quality_metrics"])

if audit["status"] == "NO_INPUT":
    print("WARNING: no ForkEvent rows in scope. Check source_date.")

# Fail the notebook (and the Job task) after recording the failure.
if audit["status"] == "FAILED":
    raise RuntimeError(f"ForkEvent pipeline FAILED: {audit['error_message']}")